# tvlog_sample_1000 → Azure AI Search Push

Fabric Lakehouse の Bronze テーブル `tvlog_sample_1000` を読み、
Azure OpenAI (`text-embedding-3-small`) で embedding を生成し、
Azure AI Search にベクトル index を push する。
さらに **integrated vectorizer** を設定して、Data Agent からの
`vector_semantic_hybrid` クエリで AI Search 側がオンザフライで
クエリ文を埋め込めるようにする。

## 処理フロー
0. パッケージインストール (`azure-search-documents`, `openai`)
1. パラメータ設定 (AI Search / AOAI / ARM ID を指定)
2. 認証クライアント初期化
3. Bronze 読み込み + paragraph ありの行に絞込
4. AI Search index を作成 (vectorizer + semantic config 付き)
5. AOAI で embedding を batch 生成 → AI Search へ batch upload
6. ベクトル検索で動作確認

## 0. パッケージインストール

Fabric Spark ランタイムに AI Search SDK が未同梱のため `%pip` で追加する。
Session ごとに 1 回実行すればよい。

In [ ]:
%pip install --quiet azure-search-documents==11.5.2 openai==1.61.0 httpx==0.27.2

## 1. パラメータ

**セキュリティ上の注意**
- `AZURE_SEARCH_ADMIN_KEY` は本番環境では直接貼り付けず、
  Azure Key Vault に格納して `notebookutils.credentials.getSecret()` 経由で取得してください。
  下のセルにはプレースホルダー `"YOUR_SEARCH_ADMIN_KEY"` があります。
- AOAI は `notebookutils.credentials.getToken()` で Entra ID トークンを取得するため key 不要です。
  実行ユーザーに AOAI リソースの **Cognitive Services OpenAI User** ロールが必要です。

### AI Search admin key を Key Vault から取得する場合
```python
AZURE_SEARCH_ADMIN_KEY = notebookutils.credentials.getSecret(
    "https://YOUR-KEYVAULT.vault.azure.net",
    "search-admin-key"
)
```

In [ ]:
# ====================================================================
# ★ ここを自分の環境に書き換えてください
# ====================================================================

BRONZE_TABLE = "tvlog_sample_1000"

# --- Azure AI Search ---
AZURE_SEARCH_ENDPOINT   = "https://YOUR-SEARCH-SERVICE.search.windows.net"
AZURE_SEARCH_INDEX_NAME = "tvlog-paragraph"
# Azure portal > AI Search > Keys > Primary admin key
# 本番では Key Vault 経由推奨 (上記セルのコメント参照)
AZURE_SEARCH_ADMIN_KEY  = "YOUR_SEARCH_ADMIN_KEY"

# --- Azure OpenAI ---
AOAI_ENDPOINT             = "https://YOUR-AOAI-RESOURCE.openai.azure.com"
AOAI_EMBEDDING_DEPLOYMENT = "text-embedding-3-small"
AOAI_API_VERSION          = "2024-08-01-preview"
EMBEDDING_DIM             = 1536

# --- バッチサイズ ---
EMBED_BATCH  = 16
UPLOAD_BATCH = 100

# ====================================================================
print(f"AOAI  : {AOAI_ENDPOINT}")
print(f"Search: {AZURE_SEARCH_ENDPOINT}")
print(f"Index : {AZURE_SEARCH_INDEX_NAME}")

## 2. 認証クライアント初期化

- AOAI: `notebookutils.credentials.getToken()` で Entra ID トークンを取得。
- AI Search: admin key (`AzureKeyCredential`)。

In [ ]:
from openai import AzureOpenAI
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient

# AOAI: Entra ID (notebookutils)
def _aoai_token() -> str:
    return notebookutils.credentials.getToken("https://cognitiveservices.azure.com")

aoai = AzureOpenAI(
    azure_endpoint=AOAI_ENDPOINT,
    api_version=AOAI_API_VERSION,
    azure_ad_token_provider=_aoai_token,
)

# AI Search: admin key
search_cred   = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)
index_client  = SearchIndexClient(endpoint=AZURE_SEARCH_ENDPOINT, credential=search_cred)
search_client = SearchClient(
    endpoint=AZURE_SEARCH_ENDPOINT,
    index_name=AZURE_SEARCH_INDEX_NAME,
    credential=search_cred,
)
print("clients ready")

## 3. AOAI 接続確認

> **事前セットアップ (ローカルで 1 回のみ)**
> integrated vectorizer 用に AI Search サービスの System-Assigned MI を有効化し、
> その MI に AOAI の `Cognitive Services OpenAI User` ロールを付与する必要があります。
> → `setup_mi_rbac.py` をローカル or Cloud Shell で実行してください。
> (Fabric notebook 上では ARM API token が取得できないため notebook 内では実行不可)

In [ ]:
_probe = aoai.embeddings.create(model=AOAI_EMBEDDING_DEPLOYMENT, input=["ping"])
print(f"AOAI reachable, dim = {len(_probe.data[0].embedding)}")
print("※ Search MI -> AOAI のロールは setup_mi_rbac.py で事前付与しておくこと")

## 4. Bronze 読み込み

In [ ]:
from pyspark.sql import functions as F

df = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("paragraph").isNotNull() & (F.col("paragraph") != "nan"))
    .select(
        F.col("para_id").cast("string").alias("para_id"),
        F.col("date").cast("string").alias("broadcast_date"),
        F.col("station").cast("string").alias("station_code"),
        F.col("program_name"),
        F.col("corner_name"),
        F.col("topic_name"),
        F.col("topic_category"),
        F.col("topic_person"),
        F.col("paragraph").alias("paragraph_text"),
    )
)

rows = [r.asDict() for r in df.collect()]
print(f"target rows: {len(rows):,}")
rows[0]

## 5. AI Search index 作成 (idempotent)

- `para_id` をキー、`paragraph_text` を検索可能テキスト、
  `paragraph_vector` (1536 dim, HNSW) をベクトル列として定義。
- **integrated vectorizer** (`AzureOpenAIVectorizer`) を設定し、
  クエリ時に AI Search 側が AOAI を自動呼び出しでベクトル化できるようにする。
- **semantic configuration** (`default`) を追加し semantic ranker を有効化。
- 同名 index があれば `create_or_update_index` で上書き (idempotent)。

In [ ]:
import inspect

from azure.search.documents.indexes.models import (
    SearchIndex, SimpleField, SearchableField, SearchField,
    SearchFieldDataType, VectorSearch, VectorSearchProfile,
    HnswAlgorithmConfiguration,
    AzureOpenAIVectorizer,
    SemanticSearch, SemanticConfiguration, SemanticPrioritizedFields, SemanticField,
)

# azure-search-documents 11.5 と 11.6+ でパラメータクラス名が異なる
try:
    from azure.search.documents.indexes.models import AzureOpenAIVectorizerParameters as _AOAIParams
    _PARAMS_KW = "parameters"
except ImportError:
    from azure.search.documents.indexes.models import AzureOpenAIParameters as _AOAIParams
    _PARAMS_KW = "azure_open_ai_parameters"

_params_sig  = inspect.signature(_AOAIParams.__init__).parameters
_RES_KW      = "resource_url"    if "resource_url"    in _params_sig else "resource_uri"
_DEPLOY_KW   = "deployment_name" if "deployment_name" in _params_sig else "deployment_id"
_vec_sig     = inspect.signature(AzureOpenAIVectorizer.__init__).parameters
_VEC_NAME_KW = "vectorizer_name" if "vectorizer_name" in _vec_sig else "name"
_prof_sig    = inspect.signature(VectorSearchProfile.__init__).parameters
_PROF_VEC_KW = "vectorizer_name" if "vectorizer_name" in _prof_sig else "vectorizer"

VECTORIZER_NAME = "aoai-vectorizer"

fields = [
    SimpleField(name="para_id",        type=SearchFieldDataType.String, key=True, filterable=True),
    SimpleField(name="broadcast_date", type=SearchFieldDataType.String, filterable=True, facetable=True),
    SimpleField(name="station_code",   type=SearchFieldDataType.String, filterable=True, facetable=True),
    SearchableField(name="program_name",   filterable=True),
    SearchableField(name="corner_name",    filterable=True),
    SearchableField(name="topic_name",     filterable=True),
    SearchableField(name="topic_category", filterable=True, facetable=True),
    SearchableField(name="topic_person",   filterable=True),
    SearchableField(name="paragraph_text", analyzer_name="ja.microsoft"),
    SearchField(
        name="paragraph_vector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=EMBEDDING_DIM,
        vector_search_profile_name="hnsw-default",
    ),
]

_aoai_params = _AOAIParams(**{
    _RES_KW:    AOAI_ENDPOINT,
    _DEPLOY_KW: AOAI_EMBEDDING_DEPLOYMENT,
    "model_name": "text-embedding-3-small",
})
vector_search = VectorSearch(
    algorithms=[HnswAlgorithmConfiguration(name="hnsw-config")],
    profiles=[
        VectorSearchProfile(
            name="hnsw-default",
            algorithm_configuration_name="hnsw-config",
            **{_PROF_VEC_KW: VECTORIZER_NAME},
        )
    ],
    vectorizers=[
        AzureOpenAIVectorizer(
            **{_VEC_NAME_KW: VECTORIZER_NAME, _PARAMS_KW: _aoai_params}
        )
    ],
)

semantic_search = SemanticSearch(
    configurations=[
        SemanticConfiguration(
            name="default",
            prioritized_fields=SemanticPrioritizedFields(
                title_field=SemanticField(field_name="program_name"),
                content_fields=[SemanticField(field_name="paragraph_text")],
                keywords_fields=[
                    SemanticField(field_name="topic_name"),
                    SemanticField(field_name="corner_name"),
                    SemanticField(field_name="topic_category"),
                ],
            ),
        )
    ]
)

index = SearchIndex(
    name=AZURE_SEARCH_INDEX_NAME,
    fields=fields,
    vector_search=vector_search,
    semantic_search=semantic_search,
)
result = index_client.create_or_update_index(index)
print(f"index ready: {result.name}")

## 6. Embedding 生成 + Upload

- AOAI を `EMBED_BATCH` 件ずつ呼んで `paragraph_text` の embedding を生成。
- 進捗を表示しながら `UPLOAD_BATCH` 件ずつ `merge_or_upload` で AI Search に送信。
- 再実行しても同じ `para_id` は上書きされるため idempotent。

In [ ]:
def embed_texts(texts):
    r = aoai.embeddings.create(model=AOAI_EMBEDDING_DEPLOYMENT, input=texts)
    return [d.embedding for d in r.data]


total    = len(rows)
uploaded = 0
buffer   = []

for i in range(0, total, EMBED_BATCH):
    chunk   = rows[i : i + EMBED_BATCH]
    vectors = embed_texts([r["paragraph_text"] for r in chunk])
    for r, v in zip(chunk, vectors):
        r["paragraph_vector"] = v
    buffer.extend(chunk)

    while len(buffer) >= UPLOAD_BATCH:
        batch, buffer = buffer[:UPLOAD_BATCH], buffer[UPLOAD_BATCH:]
        search_client.merge_or_upload_documents(documents=batch)
        uploaded += len(batch)
        print(f"  uploaded {uploaded:,} / {total:,}")

if buffer:
    search_client.merge_or_upload_documents(documents=buffer)
    uploaded += len(buffer)
    print(f"  uploaded {uploaded:,} / {total:,}")

print("\n✓ push 完了")

## 7. 動作確認 (ベクトル検索)

In [ ]:
from azure.search.documents.models import VectorizedQuery

query = "ラーメンに関する放送を教えて"
qvec  = embed_texts([query])[0]

results = search_client.search(
    search_text=None,
    vector_queries=[
        VectorizedQuery(vector=qvec, k_nearest_neighbors=5, fields="paragraph_vector")
    ],
    select=["para_id", "broadcast_date", "program_name", "topic_name", "paragraph_text"],
)
for r in results:
    print(f"[{r['@search.score']:.3f}] {r['program_name']} / {r['topic_name']}")
    print("  ", r["paragraph_text"][:120], "...")